# PDF to Vector Database Demo


In [ ]:
!pip install -q langchain-community pypdf chromadb sentence-transformers

### 1. Load the PDF Document


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Make sure you have a sample.pdf in this directory, or change the path!
try:
    loader = PyPDFLoader("sample.pdf")
    documents = loader.load()
except Exception as e:
    print("Please place a 'sample.pdf' in the same folder to run this cell. Error:", e)

### 2. Chunk the Text


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
if 'documents' in locals():
    chunks = text_splitter.split_documents(documents)
    print(f"Split the document into {len(chunks)} smaller chunks.")
    print("\nExample Chunk:")
    print(chunks[0].page_content)

### 3. Create Embeddings & Store in ChromaDB


In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
import os

if 'chunks' in locals():
    # 1. Initialize the embedding model natively via Ollama to prevent batching errors
    embedding_function = OllamaEmbeddings(
        model="nomic-embed-text:v1.5",
        base_url="https://rock0230-local.hf.space"
    )
    
    # 2. Set database directory
    DB_DIR = "./local_demo_chroma_db"
    
    # 3. Save chunks into the Vector Database
    db = Chroma.from_documents(chunks, embedding_function, persist_directory=DB_DIR)


### 4. Retrieve Context (Testing the RAG)


In [ ]:
if 'db' in locals():
    query = "What is the main topic of this document?" # Change this to test your PDF!
    
    # Search for the top 3 most relevant chunks
    results = db.similarity_search(query, k=3)
    
    print(f"Found {len(results)} relevant chunks!\n")
    for i, res in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(res.page_content)
